# Sample report (`report.ipynb`)

One run writes every figure and table to **`<paths.output>/<CASE>/samples/<OUT_NAME>/`** (a timestamp is added if it exists; nothing is overwritten), plus an `index.md` that lists every file.

1. **Metrics**: everything `evaluate` computes (paired errors with bootstrap 95% CI, own/other, W1/R0 of every event scalar, C2ST, Fréchet, memorization, pass/fail), plus voxel-set overlap (IoU / precision / recall at 192³, 96³, 48³), photon-weighted recall, centroid shift, shower-maximum shift, r50 / r90 and z-extent ratios, cells per level, box-counting dimensions, time per sample and T2 scale. No evaluate run is needed; event metrics share the evaluate / viz caches and run on `WORKERS` processes.
2. **Web-page figures**: static renders of the chain_page 3D view (same renderer and colours), its six distribution charts, and the 2D projection panels and gallery of viz.
3. **Chain intermediates** (this section runs the models; `DEVICE` is cpu or cuda): SDEdit step-by-step figure, chain overview, DiT denoising trajectory, one event at several σ, and the unconditional chain from pure noise.

Output of `selftest` / the smoke config is **synthetic data**, and figure titles say so.


In [ ]:
# parameters (run_notebook.py --set NAME=VALUE replaces these lines; lists are comma-separated strings)
CONFIG = 'configs/v2.yaml'
CASE = 'v2a'
DEVICE = 'cpu'                 # 'cuda' in the GPU job; only section 3 runs models
PAIRED_RUNS = 'sa_truth,sa_recon,sa_truth_t2off,sa_recon_t2off,sdedit_0.25,sdedit_0.5,sdedit_1,sdedit_2,sdedit_5'
FULL_RUN = 'full'              # '' = no unconditional run
V1_DIR = None                  # e.g. '<code dir>/artifacts/v1_samples/truth128' (written by jobs/v1_samples.sub)
OUT_NAME = 'report'            # -> <case>/samples/<OUT_NAME>/ (timestamped if it exists)
WORKERS = 8                    # processes for the event metrics (match request_cpus)
N_TRUTH = 256                  # test events for the truth distribution (R0, C2ST, histograms)
MAX_PAIRED = 128               # paired samples per run used for the metrics
N_FULL = 256                   # unconditional samples used for the metrics
EVENTS_3D = None               # None = first 6 events common to the paired runs
RUNS_3D = None                 # None = paired runs without t2off; if more than 5: the first 2 + 3 spread over the rest
N_GALLERY = 8
MODEL_STEPS = True             # False skips section 3 (the only part that runs models)
STEP_EVENTS = None             # None = first 2 of the 3D events
STEP_SIGMA = 1.0
MATCH_RUN = None               # e.g. 'sdedit_1': seed, σ and T2 from that run, and the result is compared with its sample
SWEEP_SIGMAS = '0.25,0.5,1,2,5'
N_FULL_TRACE = 2               # unconditional samples traced from pure noise (seeds of FULL_RUN #0, #1, ...)
DPI = 150
FORMATS = 'png'                # 'png,pdf' also writes vector copies

In [ ]:
import os, sys, time
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'sparseshower').is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from sparseshower.common import config, write_json
from sparseshower.loader import case_dir, meta_for
from sparseshower import viz, report as R
from sparseshower.evaluate import _clean, passfail_md

def _list(v, cast=str):
    if v is None or v == '':
        return []
    if isinstance(v, str):
        v = [x for x in v.split(',') if x.strip()]
    elif not isinstance(v, (list, tuple)):
        v = [v]
    return [cast(x.strip() if isinstance(x, str) else x) for x in v]

c = config(CONFIG)
LBL = viz.data_label(c)
rep = R.Report(c, CASE, OUT_NAME, dpi=DPI, formats=_list(FORMATS))
print('project :', ROOT)
print('config  :', CONFIG, '| occ_threshold', c['field']['occ_threshold'], '| data', c['paths']['processed'])
print('output  ->', rep.dir)
if c.get('synthetic'):
    print('*** SYNTHETIC data (smoke config): nothing here is CORSIKA ***')

names = _list(PAIRED_RUNS) + ([FULL_RUN] if FULL_RUN else [])
runs = R.load_runs(c, CASE, names, V1_DIR)
PAIRED = [n for n in _list(PAIRED_RUNS) if n in runs] + (['V1'] if 'V1' in runs else [])
FULL = FULL_RUN if FULL_RUN in runs else None
inv = R.inventory(runs)
for r in inv:
    print(f"{r['run']:<16} {r['chain']:<9} {r['samples']:>5} samples  σ {r['sigma']}  T2 {r['t2']}  thr {r['occ_threshold']}  "
          f"ckpt {r['steps']}  {r['sec_median']:.1f} s/sample  failed {r['failed']}")
rep.csv(inv, 'inventory', 'settings, samples, checkpoint steps, median seconds per sample, failures and median T2 scale of each run')
rep.fig(R.plot_table([dict(r, metric=r['run']) for r in inv], [k for k in inv[0] if k != 'run'] if inv else [],
                     f'Runs in this report{LBL}', fmt='{:.4g}') if inv else None, 'inventory', 'overview of all runs (same as inventory.csv)')

## 1. Metrics

- **Paired metrics** (sa_truth / sa_recon / sdedit / V1, compared with the same test event): computed per event, averaged, with a bootstrap 95% CI. `*_ratio`, `iou_*`, `precision_*`, `recall_*`, `q_recall_*` are ideally 1; `*_rel_err`, `*_l1`, `w1_*`, `*_diff`, `*_shift_*` ideally 0.
- **Distribution metrics** (every run, including full): W1 between the event-level scalar distributions of generated samples and test truth, divided by the W1 between the two truth halves (R0). ≈1 means truth-level agreement, threshold ≤ 2. C2ST AUC (0.5 is best, threshold ≤ 0.65); Fréchet distance; for full also the memorization check (nearest distance of generated latents to the training set / the same for validation latents; ≥ 0.9 means no copying of the training set).
- With very few events (e.g. a 6-event preview) the CIs are wide; C2ST needs at least 10 samples, otherwise it is empty.


In [ ]:
t0 = time.time()
summary, per_event, truth_list, gen_metrics = R.compute_summary(
    c, CASE, runs, n_truth=N_TRUTH, max_paired=MAX_PAIRED, n_full=N_FULL, workers=WORKERS)
print(f'metrics done in {time.time() - t0:.0f} s')
truth_by_eid = {}
from sparseshower.evaluate import truth_metrics
for e in {r['eid'] for r in per_event}:
    truth_by_eid[e] = truth_metrics(c, e)
write_json(rep.dir / 'summary.json', _clean(summary))
rep.items.append(('summary.json', 'all metrics (same structure as the summary.json of evaluate)'))
rep.csv(per_event, 'per_event_paired', 'per-event metrics of the paired runs (one row per run x event)')
pt = R.paired_table(summary)
dt = R.distribution_table(summary)
rep.csv(pt, 'metrics_paired', 'paired metrics: mean and 95% CI')
rep.csv(dt, 'metrics_distribution', 'distribution metrics: W1/R0, C2ST AUC, Fréchet, memorization ratio')
md_text = passfail_md(summary)
rep.text(md_text, 'passfail.md', 'pass/fail table (same format as the passfail.md of evaluate)')
print(md_text[-3000:])

In [ ]:
pn = [n for n in summary['rungs'] if 'paired' in summary['rungs'][n]]
TH = c['thresholds']
def good_paired(m, v, run):
    chain = summary['rungs'][run].get('chain')
    if m == 'q_rel_err' and chain in ('sa_truth', 'sa_recon'):
        return v <= TH['paired']['q_total_rel_err']
    if m == 'long_l1' and chain == 'sa_recon':
        return v <= TH['paired']['long_l1']
    return None
rep.fig(R.plot_table(pt, pn, f'Paired metrics vs truth: mean over events (95% CI in metrics_paired.csv){LBL}',
                     good=good_paired),
        'metrics_paired', 'paired metrics (mean); green/red = passes/fails the threshold (only Q of sa_truth / sa_recon and long_l1 of sa_recon have one)')
dn = ['R0'] + list(summary['rungs'])
def good_dist(m, v, run):
    if m.startswith('W1/R0'):
        return v <= TH['distribution']['w1_ratio']
    if m == 'C2ST AUC':
        return v <= TH['distribution']['c2st_auc']
    if m == 'memorization ratio':
        return v >= TH['memorization_ratio']
    return None
rep.fig(R.plot_table(dt, dn, f'Distribution metrics vs test truth (R0 = truth half B vs half A){LBL}', good=good_dist,
                     ),
        'metrics_distribution', 'distribution metrics; green/red = passes/fails the threshold (W1/R0 <= 2, AUC <= 0.65, memorization ratio >= 0.9)')
pf = [dict(metric=f"{r['rung']}: {r['metric']}", value=r['value'], threshold=r['threshold'], ok='yes' if r['ok'] else 'NO')
      for r in summary['passfail']]
rep.fig(R.plot_table(pf, ['value', 'threshold', 'ok'], f'Pass / fail{LBL}',
                     figsize=(9, 0.3 * len(pf) + 1.0)) if pf else None,
        'passfail', 'pass/fail table (every threshold of every run)')

In [ ]:
if pn:
    rep.fig(R.plot_paired_boxes(per_event, pn, title=f'Paired metrics per event (box = IQR, dotted = ideal){LBL}'),
            'paired_boxes', 'per-event distribution of the paired metrics (box plots), all chains in one figure')
    rep.fig(R.plot_vs_size(per_event, truth_by_eid, pn, title=f'Paired metrics vs event size{LBL}'),
            'paired_vs_size', 'paired metrics against event size')
    rep.fig(R.plot_levels(per_event, pn, title=f'Structure per level: cells and overlap with truth{LBL}'),
            'levels', 'cell-count ratio and IoU with truth at 48³ / 96³ / 192³: shows at which level errors appear')
rep.fig(R.plot_timing(runs, title=f'Time per sample and T2 scale{LBL}'), 'timing', 'time per sample and T2 scale factor')
left = next((n for n in pn if summary['rungs'][n]['chain'] == 'sa_recon'), 'sa_recon')
fig = viz.plot_sdedit(summary, left=left, right=FULL or 'full')
if fig is not None:
    fig.suptitle(f'SDEdit curve: from per-event (σ = 0) to unconditional{LBL}', fontsize=11)
    fig.tight_layout()
rep.fig(fig, 'sdedit_curve', 'SDEdit curve: paired errors, own/other, C2ST and W1/R0 from σ = 0 (sa_recon) to full')
rep.fig(viz.plot_c2st(summary), 'c2st', 'C2ST AUC with 95% CI per run (0.5 is best, threshold 0.65)')
rep.fig(viz.plot_profiles(summary, truth_list), 'profiles_mean', 'mean longitudinal and radial profiles: truth ±1σ and every run')
if FULL:
    rep.fig(viz.plot_distributions(c, gen_metrics[FULL][1], truth_list, summary, FULL),
            'distributions_full', f'event-level scalar distributions of {FULL} vs test truth, W1/R0 in the titles')

## 2. Web-page figures

- **3D**: the same renderer as the chain_page web page (light accumulated along the line of sight, viridis by log Q, shared colour scale); one figure per event, top row perspective, bottom row side view x–z.
- **Unconditional samples**: each generated sample next to the test event closest in event-feature space, **a visual reference only, not a pair**.
- **Six distribution charts**: the same log Q, nearest-neighbour distance, local linearity, longitudinal / radial profile and box-counting dimension charts as the web page, summarized from per-event quantile vectors so that each event weighs equally.


In [ ]:
common = None
for n in [p for p in PAIRED if p != 'V1']:
    ids = {int(r['eid']) for r in runs[n][0]}
    common = ids if common is None else common & ids
test_order = [int(e) for e in meta_for(c)['split']['test']]
EV3 = _list(EVENTS_3D, int) or [e for e in test_order if common and e in common][:6]
plain = [n for n in PAIRED if 't2off' not in n]
default3 = plain if len(plain) <= 5 else plain[:2] + [plain[2:][i] for i in np.unique(np.linspace(0, len(plain) - 3, 3).round().astype(int))]
R3 = [n for n in _list(RUNS_3D) if n in runs] or default3
print('3D events:', EV3, '| runs:', R3)
from sparseshower.data import load_event
qs = [load_event(Path(c['paths']['processed']), e)['q'] for e in EV3]
LQ = R.lq_range(qs, c['data']['q_eps']) if qs else (0.0, 1.0)
for e in EV3:
    rep.fig(R.plot_3d(c, R.event_panels(c, e, runs, R3), LQ, title=f'test event {e}: truth and paired chains{LBL}'),
            f'3d_event_{e}', f'event {e} in 3D: truth and the paired chains (top: perspective, bottom: side view x–z)')

In [ ]:
if FULL:
    rows, ms = gen_metrics[FULL]
    k = min(N_GALLERY, len(rows))
    near = viz.nearest_truth(c, ms[:k], test_order[:300])
    for s in range(0, k, 4):
        panels = []
        for r, t in zip(rows[s:s + 4], near[s:s + 4]):
            ev = load_event(Path(c['paths']['processed']), t)
            panels.append((f"{FULL} #{int(r['index'])}", f"{len(r['q']):,} voxels · Q {np.sum(r['q']):.3g}", r['ijk'], r['q']))
            panels.append((f'nearest test event {t}', 'visual reference, not paired', ev['ijk'], ev['q']))
        rep.fig(R.plot_3d(c, panels, LQ, title=f'unconditional samples next to the nearest test event (not paired){LBL}'),
                f'3d_gallery_{s // 4 + 1}', 'unconditional samples (left) and the nearest test event in feature space (right; visual reference, not a pair)')
if pn:
    rep.fig(R.plot_web_charts(gen_metrics, truth_by_eid, [int(r['eid']) for r in per_event if r['run'] == pn[0]], pn,
                              FULL, title=f'Web-page charts: truth vs chains (each event weighs equally){LBL}'),
            'web_charts', 'the six web-page charts: log Q, nearest-neighbour distance, local linearity, longitudinal / radial profiles, box-counting dimension')

In [ ]:
paired_rows = {n: runs[n][0] for n in R3}
for e in EV3[:3]:
    fig, table = viz.event_panel(c, e, paired_rows)
    rep.fig(fig, f'panel_event_{e}', f'event {e} 2D projection panel (x–z and y–z photon sums, 3D scatter, profiles)')
    viz.print_table(table)
if FULL:
    rows, ms = gen_metrics[FULL]
    rep.fig(viz.gallery(c, CASE, FULL, rows, ms, N_GALLERY), 'gallery_2d', '2D version of the unconditional gallery (x–z photon sums)')

## 3. Chain intermediates (runs the models)

Random generators and their order are the same as in `sample.py`: with `MATCH_RUN`, the seed of that run reproduces its sample, and the result is compared with the saved sample (bit-identical only on the same device type with the same checkpoints).

- **Step-by-step figure**: latent → noise → DiT → AE decode → start set → struct levels → attr → T2, each next to truth.
- **Chain overview**: one row generated, one row truth; fits on one slide.
- **DiT denoising**: the state x_t at a few σ, and the p_occ decoded from its estimate D(x_t).
- **σ sweep**: one event, same noise, σ = 0 (sa_recon) and SWEEP_SIGMAS.
- **Unconditional**: the whole chain from pure noise (σ = 80); the reference is the nearest test event in feature space, **not a pair**.


In [ ]:
if MODEL_STEPS:
    from sparseshower.chain_steps import plot as steps_plot
    ch = R.load_models(c, CASE, device=DEVICE)
    print('models on', ch['device'], '| checkpoints', ch['steps'])
    STEPS = _list(STEP_EVENTS, int) or EV3[:2] or test_order[:2]
    for e in STEPS:
        seed, sigma, t2 = 0, float(STEP_SIGMA), None
        if MATCH_RUN:
            try:
                m = R.run_seed(c, CASE, MATCH_RUN, eid=e)
                seed, sigma, t2 = m['seed'], float(m['sigma']), m['t2']
            except (ValueError, FileNotFoundError, KeyError) as err:
                m = None
                print(f'{MATCH_RUN}: {err}; using seed 0')
        r = R.trace_chain(ch, 'sdedit', e, sigma, seed, t2=t2)
        if MATCH_RUN and m:
            r['match'] = f"{MATCH_RUN} #{m['index']}: " + R.compare_saved(r, m['path'])
            print(r['match'])
        print(f"event {e}: seconds", {k: round(v, 1) for k, v in r['secs'].items()})
        rep.fig(steps_plot(r, c, f'SDEdit chain step by step · test event {e} · σ = {sigma:g}{LBL}'),
                f'steps_event_{e}', f'event {e} SDEdit chain step by step (σ = {sigma:g}), each step next to truth')
        rep.fig(R.plot_strip(r, c, f'SDEdit chain overview · test event {e} · σ = {sigma:g}{LBL}'),
                f'strip_event_{e}', f'event {e} chain overview: top row generated, bottom row truth')
        rep.fig(R.plot_evolution(r, c, f'DiT denoising · test event {e} · σ = {sigma:g} → 0{LBL}'),
                f'dit_event_{e}', f'event {e}: latent and decoded p_occ during DiT denoising')

In [ ]:
if MODEL_STEPS and STEPS:
    e = STEPS[0]
    res = [R.trace_chain(ch, 'sa_recon', e, 0.0, 0)]
    for s in _list(SWEEP_SIGMAS, float):
        res.append(R.trace_chain(ch, 'sdedit', e, s, 0))
    rep.fig(R.plot_sweep(res, c, f'One event, increasing σ: how much of the event survives · test event {e}{LBL}'),
            f'sigma_sweep_event_{e}', f'event {e} at σ = 0 (sa_recon) and increasing σ: larger σ moves further from the event')

In [ ]:
if MODEL_STEPS and N_FULL_TRACE:
    for k in range(int(N_FULL_TRACE)):
        seed, note = k, f'seed {k}'
        if FULL:
            m = R.run_seed(c, CASE, FULL, index=k)
            seed, note = m['seed'], f'{FULL} #{k}'
        r = R.trace_chain(ch, 'full', seed=seed)
        r = R.nearest_reference(ch, r)
        if FULL:
            r['match'] = R.compare_saved(r, m['path'])
            print(note + ':', r['match'])
        print(f'{note}: seconds', {kk: round(v, 1) for kk, v in r['secs'].items()})
        rep.fig(R.plot_strip(r, c, f'Unconditional chain from pure noise · {note} · lower row: nearest test event {r.get("ref_eid")} (not paired){LBL}'),
                f'strip_full_{k}', f'unconditional chain ({note}); bottom row: nearest test event in feature space, visual reference only')
        rep.fig(R.plot_evolution(r, c, f'DiT from pure noise · {note}{LBL}'),
                f'dit_full_{k}', f'unconditional chain ({note}): DiT denoising from pure noise')

In [ ]:
idx = rep.index(f'v2 sample report · case {CASE}{LBL}')
print('figures and tables in', rep.dir)
print(idx.read_text())